# Семинар 06. Применяем сортировку

Сортировка нужна не только для того, чтобы вывести числа по возрастанию. Сегодня получим удобный каталог, построим очередь по заданным правилам, найдём близкие измерения и соберём разрозненные периоды занятости в общий календарь.

Примерно **80% занятия — применение сортировки**, задачи 1–4. Оставшиеся **20% — реализация одного алгоритма**, задача 5. В первых четырёх задачах используем `sorted()` и `.sort()`. Переписывать их устройство для каждой задачи не требуется.

## Цели

- выбирать между `sorted()` и `.sort()`, помня, какой объект меняется и что возвращается;
- готовить ключ: число из строки, несколько полей, внешний порядок групп;
- задавать разные направления и опираться на стабильность, в том числе несколькими проходами;
- использовать упорядоченный список как инструмент: соседи вместо всех пар, границы вместо отрезков;
- реализовать сортировку подсчётом и назвать условия, при которых она применима.

## Перед началом

Откройте [лекцию](lecture.ipynb). Для задач нужны списки, словари, функции, циклы и `sorted()`
с `key`. Новые библиотеки не понадобятся.

Начнём с разминки: четыре фрагмента, к каждому сначала записываем прогноз. Первую задачу
разберём вместе: обсудим ожидаемый результат и напишем ключ в коде. Задания 2 и 3 решайте
самостоятельно, обсуждая затруднения. Четвёртую разберём совместно у доски или в онлайн-коде:
она показывает, как сортировка помогает дальнейшему проходу по данным. Затем соберём
собственную сортировку подсчётом. Дополнительное задание — сортировка слиянием — для тех, кто
закончил раньше.

В ноутбуке оставлены заготовки без решений. Сначала запустите ячейку с определением функции,
затем ячейку проверок под ней. До появления решения проверки намеренно не проходят — запускать весь
незаполненный ноутбук подряд не нужно. Подсказки к части задач скрыты под раскрывающимися
блоками; сначала попробуйте предложить свой способ.

Во всех задачах вход соответствует условию. Проверки формата и обработка исключений не нужны.
Функции возвращают результат; исходные списки, словари и списки правил изменять нельзя. Глубокое
копирование записей не требуется. Под «новым списком» понимается отдельный внешний список, в том
числе для пустого входа.

## Разминка. Что напечатает

Четыре фрагмента, каждый запускается отдельно. Сначала запишите прогноз, затем выполните ячейку
и сверьтесь. Разбор — под спойлером после всех фрагментов.

In [ ]:
# Фрагмент 1
values = [30, 10, 20]
alias = values
ordered = sorted(values, reverse=True)
returned = values.sort()

print("Исходный список:", values)
print("Второе имя:", alias)
print("Отдельный результат:", ordered)
print("Возврат sort:", returned)

In [ ]:
# Фрагмент 2
pages = ["10", "9", "007"]

print(sorted(pages))
print(sorted(pages, key=int))

In [ ]:
# Фрагмент 3
arrivals = [("A", 80), ("B", 95), ("C", 80), ("D", 95)]

print([record[0] for record in sorted(arrivals, key=lambda record: record[1], reverse=True)])
print([record[0] for record in reversed(sorted(arrivals, key=lambda record: record[1]))])

In [ ]:
# Фрагмент 4
staff = [
    ("Omsk", "Boris", 95),
    ("Kazan", "Anna", 80),
    ("Omsk", "Gleb", 95),
    ("Kazan", "Vera", 70),
]

by_score = sorted(staff, key=lambda row: row[2])
both = sorted(by_score, key=lambda row: row[0], reverse=True)
print([row[1] for row in both])

<details>
<summary>Разбор</summary>

**Фрагмент 1.** `[10, 20, 30]`, `[10, 20, 30]`, `[30, 20, 10]`, `None`. `sorted()` построил
новый список, а `.sort()` упорядочил существующий объект на месте и вернул `None` — поэтому
`values = values.sort()` потеряло бы список целиком. Имя `alias` ведёт к тому же объекту, что и
`values`, и видит изменение. Если нужен только один наибольший элемент, полная сортировка
лишняя: хватит `max()`.

**Фрагмент 2.** `['007', '10', '9']`, затем `['007', '9', '10']`. Строки сравниваются
посимвольно по кодам: у `"007"` первый символ `'0'` меньше всех, а `"10"` оказывается раньше
`"9"`, потому что `'1'` меньше `'9'`. С `key=int` каждая строка один раз превращается в число, и
порядок становится числовым. Сами строки при этом не меняются — ключ только для сравнения.

**Фрагмент 3.** `['B', 'D', 'A', 'C']` и `['D', 'B', 'C', 'A']`. Оба способа дают убывание по
баллам, но по-разному поступают с равными: `reverse=True` сохраняет порядок поступления внутри
группы, а переворот готового списка меняет его на обратный. Когда в условии сказано «при равных
баллах порядок поступления», подходит только первый.

**Фрагмент 4.** `['Boris', 'Gleb', 'Vera', 'Anna']`. Сначала записи упорядочены по баллам, затем
стабильная сортировка по городу переставила группы, не разрушив порядок внутри них. Так
получается сортировка по нескольким полям с **разными** направлениями: проходы идут от менее
важного критерия к более важному, последний проход — по главному.

</details>

## Задание 1. Каталог книг: данные и ключ

Каталог пришёл из файла. У каждой книги есть уникальный `id`, название `title` и число страниц `pages`, записанное **строкой**. Ведущие нули допустимы. Названия могут отличаться регистром и иметь пробелы по краям.

Реализуйте `order_books(books)`, возвращающую новый список записей:

1. Сначала книги с меньшим **числом** страниц.
2. При равном числе страниц — название по возрастанию после `strip()` и `casefold()`.
3. При равенстве обоих ключей — прежний порядок записей.

Названия и строки с числом страниц в результате должны остаться в исходном виде. До 100 000 книг, число страниц от 0 до 10 000; строка `pages` не длиннее 10 символов, название — 100. При этих ограничениях достаточно O(n log n) времени.

Начнём вместе с порядка только по числу страниц. Затем добавьте название. Для функции ключа сначала используйте `def`; после проверки попробуйте короткую запись через `lambda`. Объясните, почему `id` не входит в ключ.

In [ ]:
books = [
    {"id": "long", "title": "Длинная книга", "pages": "100"},
    {"id": "fruit", "title": "Яблоко", "pages": "20"},
    {"id": "z-first", "title": " азбука ", "pages": "020"},
    {"id": "a-second", "title": "АЗБУКА", "pages": "20"},
    {"id": "short", "title": "Короткая книга", "pages": "3"},
]


def order_books(books):
    # TODO: верните записи в требуемом порядке, не изменяя их поля.
    raise NotImplementedError

In [ ]:
before = [book.copy() for book in books]
result = order_books(books)
assert [book["id"] for book in result] == [
    "short", "z-first", "a-second", "fruit", "long"
]
assert result is not books
assert books == before
assert result[1]["title"] == " азбука "
assert result[1]["pages"] == "020"
empty = []
empty_result = order_books(empty)
assert empty_result == [] and empty_result is not empty

После решения измените задачу: нужны только три книги с наименьшим числом страниц с теми же правилами равенства. Как получить их из готового результата? А если нужна только одна книга с наименьшим ключом — обязательно ли сортировать весь каталог?

Проверьте, что в вашей реализации `"100"` сравнивается с `"20"` как число, а `"020"` и `"20"` дают одинаковый числовой ключ. Две «азбуки» должны сохранить исходный порядок, хотя их `id` расположены в обратном алфавитном порядке.

## Задание 2. Очередь: порядок задаётся правилами

Заявки поступают в порядке регистрации. Каждая — словарь с уникальным строковым `id` и полем `status`. Порядок статусов передан отдельно, например `['urgent', 'normal', 'later']`. Алфавит здесь не помогает.

Реализуйте `order_queue(requests, status_order)`. Верните новый список заявок: группы идут в порядке из `status_order`, внутри каждой группы сохраняется порядок поступления. Список статусов не содержит повторов, каждый статус заявки в нём есть. Для пустого списка заявок список статусов тоже может быть пустым.

До 100 000 заявок и до 10 статусов; длина каждого статуса не больше 20 символов. Используйте стабильность встроенной сортировки. Желаемая оценка — O(n log n + k), где `k` — число статусов. Неизменность входных записей и самого `status_order` — часть условия.

Сначала объясните словами, какое значение ключа должно соответствовать каждому статусу. После решения поменяйте порядок статусов и проверьте, что код функции менять не пришлось.

In [ ]:
requests = [
    {"id": "a", "status": "later"},
    {"id": "z", "status": "urgent"},
    {"id": "b", "status": "normal"},
    {"id": "a2", "status": "urgent"},
    {"id": "c", "status": "normal"},
]
status_order = ["urgent", "normal", "later"]


def order_queue(requests, status_order):
    # TODO: используйте переданный порядок статусов и сохраните порядок внутри групп.
    raise NotImplementedError

In [ ]:
before = [request.copy() for request in requests]
order_before = status_order.copy()
result = order_queue(requests, status_order)
assert [request["id"] for request in result] == ["z", "a2", "b", "c", "a"]
assert [request["id"] for request in order_queue(
    requests, ["later", "normal", "urgent"]
)] == ["a", "b", "c", "z", "a2"]
assert result is not requests
assert requests == before and status_order == order_before
empty = []
empty_result = order_queue(empty, [])
assert empty_result == [] and empty_result is not empty

Должна ли заявка `a2` обогнать `z` внутри срочной группы? Что произойдёт, если добавить `id` в ключ «для определённости»? Учитывает ли программа требования задачи или случайно придумывает дополнительное правило?

## Задание 3. Город по убыванию, имя по возрастанию

Отдел кадров прислал сотрудников в порядке приёма. Каждая запись — словарь с полями `city` и
`name`, обе строки непустые, из латинских букв.

Реализуйте `order_by_city(staff)`. Верните новый список записей:

1. Сначала города по **убыванию**: сравниваем строки как обычно, по кодам символов.
2. Внутри города — имена по **возрастанию**.
3. Если обе строки совпали, сохраняется порядок приёма.

До 100 000 записей, длина каждой строки не больше 20 символов. Сами записи менять нельзя,
исходный список тоже. Достаточно O(n log n).

Одним кортежным ключом это не записывается: к числу можно применить минус и перевернуть его
порядок, а к строке — нет. Подумайте, как получить два разных направления, опираясь на
стабильность встроенной сортировки.

In [ ]:
staff = [
    {"city": "Kazan", "name": "Vera"},
    {"city": "Omsk", "name": "Boris"},
    {"city": "Kazan", "name": "Anna"},
    {"city": "Omsk", "name": "Gleb"},
]


def order_by_city(staff):
    # TODO: верните записи в требуемом порядке, не изменяя их поля.
    raise NotImplementedError

In [ ]:
before = [person.copy() for person in staff]
result = order_by_city(staff)
assert [(person["city"], person["name"]) for person in result] == [
    ("Omsk", "Boris"), ("Omsk", "Gleb"), ("Kazan", "Anna"), ("Kazan", "Vera")
]
assert result is not staff
assert staff == before
empty = []
empty_result = order_by_city(empty)
assert empty_result == [] and empty_result is not empty
assert [person["name"] for person in order_by_city([
    {"city": "A", "name": "Zoe"}, {"city": "A", "name": "Ada"}
])] == ["Ada", "Zoe"]
twins = [{"city": "A", "name": "X"}, {"city": "A", "name": "X"}]
ordered_twins = order_by_city(twins)
assert ordered_twins[0] is twins[0] and ordered_twins[1] is twins[1]
assert [person["city"] for person in order_by_city([
    {"city": "omsk", "name": "A"}, {"city": "Omsk", "name": "B"}
])] == ["omsk", "Omsk"]

В каком порядке идут ваши проходы и что будет, если поменять их местами? Последняя проверка
показывает, что `"omsk"` оказывается раньше `"Omsk"`: строчные буквы имеют бо́льшие коды, а
порядок по городу — убывающий.

Сколько проходов понадобится, если полей станет четыре, а направления будут приходить списком?

## Задание 4. Сколько нужно переговорок

Для одного этажа поступили брони из разных источников. Бронь — кортеж `(start, end)` с временем
в минутах от начала дня: `0 <= start < end <= 1440`. Начало включается, конец — нет, поэтому
брони `(10, 20)` и `(20, 30)` можно провести в одной комнате. Записи могут пересекаться,
повторяться и находиться в любом порядке.

Реализуем `rooms_needed(bookings)`: наименьшее число переговорок, при котором все брони пройдут
и никакие две пересекающиеся не окажутся в одной комнате. Для пустого списка ответ `0`. Исходный
список не меняем.

До 200 000 броней; достаточно O(n log n) времени. Встроенная сортировка разрешена. Разберём вход
`[(10, 30), (25, 45), (40, 60)]`: ответ `2`.

<details>
<summary>Вопросы к условию</summary>

Нарисуем брони отрезками друг под другом. В какой момент дня комнат нужно больше всего? Какое
событие увеличивает число занятых комнат, а какое уменьшает? Нужно ли при этом знать, **какая**
бронь в какой комнате, — или достаточно считать события? Обязательно ли держать начала и концы
вместе в одном кортеже?

</details>

In [ ]:
def rooms_needed(bookings):
    # TODO: верните наименьшее число переговорок.
    raise NotImplementedError

In [ ]:
bookings = [(10, 30), (25, 45), (40, 60)]
before = bookings.copy()
assert rooms_needed(bookings) == 2
assert bookings == before
assert rooms_needed([]) == 0
assert rooms_needed([(0, 1440)]) == 1
assert rooms_needed([(10, 20), (20, 30)]) == 1
assert rooms_needed([(10, 20), (19, 30)]) == 2
assert rooms_needed([(0, 10), (10, 20), (20, 30)]) == 1
assert rooms_needed([(0, 100), (10, 20), (30, 40)]) == 2
assert rooms_needed([(0, 10), (0, 10), (0, 10)]) == 3
assert rooms_needed([(5, 15), (0, 10), (12, 20)]) == 2

Проверьте на своём решении: три одинаковые брони требуют три комнаты, а цепочка
`(0, 10), (10, 20), (20, 30)` — одну. Что изменится в ответе, если конец брони считать
включённым?

Сравните с задачей о ближайших измерениях: там сортировка превратила перебор пар в проход по
соседям, здесь — перебор моментов времени в проход по событиям.

## Задание 5. Своя сортировка: подсчёт

На этот блок оставляем примерно пятую часть занятия. Вспомним идею из лекции у доски или на нескольких значениях в коде, затем самостоятельно реализуйте `counting_sort(values, max_value)`.

Вход содержит целые числа от `0` до `max_value` включительно. Верните **новый** список по возрастанию. Сохраните все повторы и исходный список. До 200 000 элементов, `0 <= max_value <= 100_000`. Вход корректен: дополнительных проверок и `raise` не требуется.

В реализации нельзя использовать `sorted()` и `.sort()`. В проверках они разрешены как эталон. Требования: O(n + k) времени и O(k) дополнительной памяти сверх результата, где `k = max_value + 1`.

Что хранится по индексу `3` в массиве счётчиков? Для какого входа массив счётчиков станет слишком большим, даже если самих чисел мало?

In [ ]:
def counting_sort(values, max_value):
    # TODO: посчитайте значения и восстановите упорядоченный список.
    raise NotImplementedError

In [ ]:
cases = [[], [0], [3, 0, 3, 1], [1, 2, 3], [3, 2, 1], [2, 2, 2]]
for values in cases:
    before = values.copy()
    result = counting_sort(values, 3)
    assert result == sorted(values)
    assert result is not values
    assert values == before
assert counting_sort([0, 0], 0) == [0, 0]
assert counting_sort([100_000, 0, 100_000], 100_000) == [0, 100_000, 100_000]

## Дополнительно. Сортировка слиянием

Задание для тех, кто закончил раньше: разбирать его на занятии не обязательно.

Сортировка подсчётом обошлась без сравнений, но потребовала небольшого диапазона значений.
Здесь собираем сортировку, которая сравнивает элементы и работает за `O(n log n)` при любых
значениях. Делаем её в два приёма.

**Сначала `merge(left, right)`.** Оба списка уже упорядочены по неубыванию. Верните новый
список из всех их элементов, тоже упорядоченный. Если очередные элементы равны, первым идёт
элемент из `left` — так слияние остаётся стабильным. Нужно `O(len(left) + len(right))`: каждый
элемент просматривается один раз, повторных проходов и поиска минимума быть не должно.

**Затем `merge_sort(values)`.** Верните новый упорядоченный список: разделите вход пополам,
упорядочьте половины тем же способом и слейте их готовой `merge`. Список из одного элемента и
пустой уже упорядочены.

В обеих функциях нельзя использовать `sorted()` и `.sort()`; в проверках они разрешены как
эталон. Входные списки не меняются. До 100 000 чисел.

<details>
<summary>Вопросы к условию</summary>

Почему в `merge` достаточно одного прохода, если оба входа упорядочены? Что произойдёт, если
при равных элементах брать из `right`, — ответ изменится, а что сломается? Сколько уровней
деления пополам у списка из 100 000 элементов и сколько работы на каждом уровне?

</details>

In [ ]:
def merge(left, right):
    # TODO: слейте два упорядоченных списка в один за один проход.
    raise NotImplementedError


def merge_sort(values):
    # TODO: разделите пополам, упорядочьте половины и слейте их через merge.
    raise NotImplementedError

In [ ]:
assert merge([], []) == []
assert merge([1, 3], []) == [1, 3]
assert merge([], [2, 4]) == [2, 4]
assert merge([1, 3, 5], [2, 4]) == [1, 2, 3, 4, 5]
assert merge([1, 1], [1]) == [1, 1, 1]
assert merge([-5, 0], [-7, 2]) == [-7, -5, 0, 2]
left_part = [1, 2]
right_part = [3]
assert merge(left_part, right_part) == [1, 2, 3]
assert left_part == [1, 2] and right_part == [3]
first = (1, "same")
second = tuple(list(first))
assert first == second and first is not second
assert merge([first], [second])[0] is first

assert merge_sort([]) == []
assert merge_sort([7]) == [7]
assert merge_sort([2, 1]) == [1, 2]
assert merge_sort([3, 2, 1]) == [1, 2, 3]
assert merge_sort([1, 2, 3]) == [1, 2, 3]
assert merge_sort([2, 2, 1, 2]) == [1, 2, 2, 2]
assert merge_sort([0, -7, 4, -7, 4]) == [-7, -7, 0, 4, 4]
values = [5, 1, 4, 1, 3]
before = values.copy()
result = merge_sort(values)
assert result == sorted(values)
assert values == before and result is not values
long_case = [(index * 37) % 101 for index in range(101)]
assert merge_sort(long_case) == sorted(long_case)

Сравните две свои сортировки: подсчёт обходится без сравнений, но требует маленького диапазона
значений; слияние сравнивает элементы и не зависит от диапазона. Какую из них вы выберете для
списка из миллиона чисел от 0 до 5? А для миллиона произвольных целых?

Ещё один вопрос, к которому мы вернёмся в домашней работе: в момент, когда `merge` забирает
элемент из `right`, что известно про все элементы, оставшиеся в `left`?

## Итоги

В каталоге мы подготовили ключ, не изменив сами записи. В очереди задали внешний порядок групп и
сохранили порядок поступления. В списке сотрудников получили два разных направления двумя
стабильными проходами. В переговорках сортировка сократила перебор моментов дня до прохода по
событиям. В последнем задании мы собрали алгоритм сами: сортировка подсчётом обходится без
единого сравнения, но требует небольшого диапазона значений. Дополнительное задание показывает
вторую сторону: слияние сравнивает элементы, зато на значения не оглядывается.

Для каждого решения объясните: зачем здесь порядок, по чему сортируем и что делаем с данными
после сортировки? Затем проверьте повторы, пустой вход и неизменность исходных данных.

## Домашняя работа

[Основные задачи](tasks.md) — три Easy и две Medium. Для желающих есть отдельная необязательная задача на **radix sort**: стабильные проходы по разрядам неотрицательных целых чисел. Идея и необходимые правила описаны в условии; на семинаре реализовывать ещё один алгоритм не нужно.

> В материалах курса приведены **предполагаемые списки домашних задач**. Домашние задания нужно решать и сдавать в **Яндекс.Контесте**. Актуальный список задач, условия и требования к оформлению находятся в Контесте. **При расхождениях информация в Контесте имеет приоритет.** Ссылки на контесты публикуются на **вики курса**.